In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## input

In [ ]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [ ]:
control_key = "is_control"
condition_rep_keys = "perturbation_embeddings"
condition_combined_keys = "condition_combined"
mass_deduct_keys = "bc1_well" # or None
random_seed = 42

condition_keys = "cytokine" # 数据集perturbation所对应的obs列名
dataset_name = "PBMC_donor11_e5test3"
sample_rep = "X_pca" #"X_scVI"  "X_flatvi" "X_state"

cov_config = {
    "donor": {
        "type": "categorical",
        "control_ot": "groupwise",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_source": "both",
        "contain_in_condition": True,
        "condition_source": "both",
    },
    "cell_type": {
        "type": "categorical",
        "control_ot": "global",
        "perturbed_ot": "global",
        "use_in_model": True,
        "model_source": "control",
        "contain_in_condition": False,
        "condition_source": None,
    },
}


if_adata_ref = None #用于根据一个参考adata快速构建pca
adata_ref_path = "data/processed/PBMC2000_pca_rep_0.2_42.h5ad"


#condition_rep_dict = pd.read_pickle("./data/processed/PBMC_cytokines.pkl")
condition_rep_dict = pd.read_pickle("./data/processed/condition_embedding_e5_test5.pkl")
condition_rep_dict = {
    k: (v["embedding"] if isinstance(v, dict) and "embedding" in v else None)
    for k, v in condition_rep_dict.items()
}

In [ ]:
filePath = './data/raw/PBMC_donor11_hvg.h5ad'
adata = sc.read_h5ad(filePath)
adata.uns["cov_config"] = cov_config
print(adata)

In [ ]:
adata.obs[control_key] = (adata.obs[condition_keys] == "PBS")
condition_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

## splitting

In [ ]:
#adata = adata[~adata.obs[condition_keys].isin(["LT-alpha2-beta1","IFN-lambda2","IFN-lambda3","IL-18Ra","LT-alpha1-beta2"])]

In [ ]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.2
condition_list = list(condition_list)
zero_shot = True

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    pert_mask = adata.obs[control_key] == False
    y = adata.obs.loc[pert_mask, condition_keys].astype(str).values
    pert_indices = np.flatnonzero(pert_mask)

    train_idx, test_idx = train_test_split(
        pert_indices,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    adata_train.uns["normalized_m"] = 1 / (1-test_ratio)
    adata_test.uns["normalized_m"] = 1 / test_ratio
    adata_control.uns["normalized_m"] = 1 
    print(condition_list)
else:
    # 按condition分割 zeroshot
    n_test = max(1, int(len(condition_list) * test_ratio))
    test_condition = rng.choice(condition_list, size=n_test, replace=False).tolist()
    print(test_condition)
    train_condition = [g for g in condition_list if g not in test_condition]
    print(train_condition)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_condition是 PBS
    adata_train = adata[adata.obs[condition_keys].isin(train_condition)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_condition)].copy()
    adata_train.uns["normalized_m"] = 1
    adata_test.uns["normalized_m"] = 1
    adata_control.uns["normalized_m"] = 1 

In [ ]:
del adata

## latent embedding

In [ ]:
n_comps = 100
n_hidden = 1024
n_layers = 2

model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [ ]:
if if_adata_ref:
    adata_ref = sc.read_h5ad(adata_ref_path,backed='r')
else:
    adata_ref = None

In [ ]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_ref = adata_ref,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    condition_combined_keys = condition_combined_keys,
    cov_config = cov_config,
    condition_rep_dict = condition_rep_dict,
    pca_method = "scanpy", # "parse"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

In [ ]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep}_{n_comps}_{if_adata_ref}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [ ]:
print(preprocess_save_path)
print(adata_control)
print(adata_train)
print(adata_test)

In [ ]:
adata_control.uns

In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()